In [1]:
import math
import os
import re
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from pathlib import Path
from gensim.models import Word2Vec

In [2]:
# ------------------ Load & Preprocess ------------------
# 1) Define base directory
BASE = Path(".").resolve() 
# 2) Define files directly
PROCESSED_DATA_FILE = BASE / r"data\processed\products_clean.parquet"
LABELS_FILE         = BASE / r"data\raw\validation_labels.csv"
INDEX_FILE          = BASE / r"data\index\inverted_index.json"

# 3) Sanity checks
print("processed:", PROCESSED_DATA_FILE.exists(), PROCESSED_DATA_FILE)
print("labels   :", LABELS_FILE.exists(), LABELS_FILE)
os.makedirs(INDEX_FILE.parent, exist_ok=True)

# 4) Load data 
try:
    df = pd.read_parquet(PROCESSED_DATA_FILE)
except Exception as e:
    print("pandas read_parquet failed:", e)
    import pyarrow.dataset as ds
    df_ = ds.dataset(PROCESSED_DATA_FILE, format="parquet").to_table().to_pandas()

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

df["tokens"] = df["title"].apply(tokenize)
df["len"] = df["tokens"].apply(len)

# Stats for BM25
N = len(df)
df_counts = Counter(t for tokens in df["tokens"] for t in set(tokens))
avgdl = df["len"].mean()


processed: True C:\Users\david\IRWA_Definitivo\IRWA_Final_Project\project_progress\part_3\data\processed\products_clean.parquet
labels   : True C:\Users\david\IRWA_Definitivo\IRWA_Final_Project\project_progress\part_3\data\raw\validation_labels.csv
pandas read_parquet failed: Repetition level histogram size mismatch


OSError: Repetition level histogram size mismatch

In [ ]:
#--------------------WORD2VEC REPREESENTATIONS----------------
# 1) Prepare training corpus (list of token lists)
corpus_tokens = df["tokens"].tolist()
pids = df["pid"].tolist()

# 2) Train Word2Vec model
w2v_model = Word2Vec(
    sentences=corpus_tokens,
    vector_size=100,   # can be 150 if you want higher quality
    window=5,
    min_count=2,
    workers=4,
    sg=1               # skip-gram (better for semantic retrieval)
)

# 3) Function to average word vectors
def average_vector(tokens, model):
    vectors = []
    for t in tokens:
        if t in model.wv:
            vectors.append(model.wv[t])
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# 4) Build document vectors for ALL documents
doc_vectors = {}

for pid, tokens in zip(pids, corpus_tokens):
    doc_vectors[pid] = average_vector(tokens, w2v_model)

# 5) Cosine similarity for dense vectors
def cosine_sim(a, b):
    if norm(a) == 0 or norm(b) == 0:
        return 0
    return np.dot(a, b) / (norm(a) * norm(b))

# 6) Word2Vec ranking function
def word2vec_rank(query):
    q_tokens = tokenize(query)
    q_vec = average_vector(q_tokens, w2v_model)

    scores = []
    for pid, d_vec in doc_vectors.items():
        sim = cosine_sim(q_vec, d_vec)
        scores.append((pid, sim))

    scores_sorted = sorted(scores, key=lambda x: x[1], reverse=True)
    return scores_sorted[:20]   # top-20 required by assignment

In [ ]:
# ------------------ IDF for TF-IDF ------------------
def idf(term):
    return math.log((N + 1) / (df_counts.get(term, 0) + 1)) + 1

# Build TF-IDF vectors per document (normalized)
tfidf_vecs = {}
for _, row in df.iterrows():
    doc_id = row["pid"]
    c = Counter(row["tokens"])
    vec = {t: c[t] * idf(t) for t in c}
    norm = math.sqrt(sum(v*v for v in vec.values()))
    if norm > 0:
        vec = {k: v/norm for k, v in vec.items()}
    tfidf_vecs[doc_id] = vec

# Cosine for sparse TF-IDF
def cosine(qvec, dvec):
    return sum(qvec[t] * dvec.get(t, 0) for t in qvec)    

In [ ]:
# ------------------ BM25 ------------------
k1, b = 1.5, 0.75

def bm25(query_terms, doc_id):
    score = 0.0
    dl = df.loc[df["pid"] == doc_id, "len"].iloc[0]
    for term in query_terms:
        f = tfidf_vecs[doc_id].get(term, 0)
        if f == 0: continue
        df_t = df_counts.get(term, 0)
        idf_bm = math.log((N - df_t + 0.5) / (df_t + 0.5) + 1e-9)
        denom = f + k1 * (1 - b + b * dl / avgdl)
        score += idf_bm * f * (k1 + 1) / (denom + 1e-9)
    return score



In [ ]:
# ------------------ Custom Score ------------------
# TF-IDF + Title Boost + Normalized Relevance (domain-based scoring)
def custom_score(query_terms, doc_id):
    # TF-IDF cosine
    c = Counter(query_terms)
    qvec = {t: c[t]*idf(t) for t in c}
    norm = math.sqrt(sum(v*v for v in qvec.values()))
    if norm > 0:
        qvec = {k: v/norm for k, v in qvec.items()}
    tfidf_score = cosine(qvec, tfidf_vecs[doc_id])

    # Title boost if term appears in title
    title = df.loc[df["pid"] == doc_id, "tokens"].iloc[0]
    title_boost = sum(1 for t in query_terms if t in title)

    # Use relevance column as a document quality hint (if available)
    rel = float(df.loc[df["pid"] == doc_id, "relevance"].iloc[0])
    rel_bonus = math.log(1 + rel)

    return tfidf_score + 0.5*title_boost + 0.3*rel_bonus

In [ ]:
# ------------------ Ranking Function ------------------
def rank_query(query_id):
    subset = df[df["query_id"] == query_id]
    query = subset["query"].iloc[0]
    q_terms = tokenize(query)

    # Build TF-IDF query
    qc = Counter(q_terms)
    qvec = {t: qc[t]*idf(t) for t in qc}
    norm = math.sqrt(sum(v*v for v in qvec.values()))
    if norm > 0:
        qvec = {k: v/norm for k, v in qvec.items()}

    # Score all docs under same query_id
    tfidf_scores = []
    bm25_scores = []
    custom_scores = []

    for _, row in subset.iterrows():
        doc_id = row["pid"]

        tfidf_scores.append((doc_id, cosine(qvec, tfidf_vecs[doc_id])))
        bm25_scores.append((doc_id, bm25(q_terms, doc_id)))
        custom_scores.append((doc_id, custom_score(q_terms, doc_id)))

    return {
        "TF-IDF": sorted(tfidf_scores, key=lambda x: x[1], reverse=True),
        "BM25": sorted(bm25_scores, key=lambda x: x[1], reverse=True),
        "Custom": sorted(custom_scores, key=lambda x: x[1], reverse=True),
    }


In [ ]:
#Testing
results = rank_query(query_id=1.0)

print("TF-IDF Top 5:", results["TF-IDF"][:5])
print("BM25 Top 5:", results["BM25"][:5])
print("Custom Top 5:", results["Custom"][:5])


In [ ]:
# -------------WORD2VEC RESULTS FOR PART 2 QUERIES----------------

queries_part2 = [
    "long sleeve denim jacket blue",
    "cotton shirt man regular fit",
    "women western wear cotton",
    "machine wash suitabl woman",
    "brand blend fabric shirt"
]

for q in queries_part2:
    print(f"\nTop-20 Word2Vec results for query: {q}")
    print("-" * 60)
    results = word2vec_rank(q)
    print(results)